# Preprocessing Pipeline

This notebook demonstrates the full MAESTRO preprocessing pipeline:
1. Load a MIDI file via `pretty_midi`
2. Convert to a binarised piano roll (128 pitch × T time-steps)
3. Segment into fixed-length windows (64 steps)
4. Convert piano-roll segments to event tokens (for Task 3)
5. Save processed splits as `.npy` files in `data/processed/`

The clean functions live in `src/preprocessing/`. This notebook calls them directly.

In [ ]:
import sys
from pathlib import Path

# Make sure the project root is on sys.path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import pretty_midi

from src.config import FS, SEQ_LEN, MAESTRO_DIR, PROCESSED_DATA_DIR
from src.preprocessing.midi_parser import load_midi
from src.preprocessing.piano_roll import midi_to_piano_roll, segment_piano_roll
from src.preprocessing.tokenizer import piano_roll_sequence_to_event_tokens, VOCAB_SIZE

print(f"Project root : {PROJECT_ROOT}")
print(f"FS={FS}, SEQ_LEN={SEQ_LEN}, VOCAB_SIZE={VOCAB_SIZE}")

## Step 1 — Load a MIDI File

In [ ]:
# Pick any file from the MAESTRO dataset
midi_files = sorted(MAESTRO_DIR.rglob("*.midi")) + sorted(MAESTRO_DIR.rglob("*.mid"))
if not midi_files:
    raise FileNotFoundError(f"No MIDI files found under {MAESTRO_DIR}")

sample_path = midi_files[0]
print(f"Using: {sample_path.name}")
midi = load_midi(str(sample_path))
print(f"Duration: {midi.get_end_time():.1f}s  |  Instruments: {len(midi.instruments)}")

## Step 2 — Piano Roll Conversion

`midi_to_piano_roll` calls `pretty_midi.get_piano_roll(fs=FS)` and binarises the result.
Shape: `(128, T)` where T is the number of time steps.

In [ ]:
roll = midi_to_piano_roll(midi, fs=FS)   # shape (128, T)
print(f"Piano roll shape: {roll.shape}  |  Active pitches: {roll.sum(axis=1).astype(bool).sum()}")

plt.figure(figsize=(14, 4))
plt.imshow(roll, aspect="auto", origin="lower", cmap="gray_r", interpolation="nearest")
plt.xlabel("Time steps")
plt.ylabel("MIDI pitch")
plt.title("Piano Roll (first 512 steps)")
plt.xlim(0, min(512, roll.shape[1]))
plt.tight_layout()
plt.show()

## Step 3 — Segmentation

`segment_piano_roll` slices the roll into non-overlapping windows of `SEQ_LEN` steps.
Segments that don't fill a full window are discarded.

In [ ]:
segments = segment_piano_roll(roll, window=SEQ_LEN)   # list of (128, SEQ_LEN)
print(f"Total segments: {len(segments)}  |  Segment shape: {segments[0].shape if segments else 'N/A'}")

# Show the first segment
if segments:
    plt.figure(figsize=(8, 4))
    plt.imshow(segments[0], aspect="auto", origin="lower", cmap="gray_r", interpolation="nearest")
    plt.xlabel("Time steps")
    plt.ylabel("MIDI pitch")
    plt.title(f"Segment 0  (shape {segments[0].shape})")
    plt.tight_layout()
    plt.show()

## Step 4 — Event Tokenisation (Task 3)

`piano_roll_sequence_to_event_tokens` encodes one segment as a fixed-length integer sequence.
Format per note: `[time_shift]* note_on velocity duration note_off`.

In [ ]:
if segments:
    seg_transposed = segments[0].T   # (SEQ_LEN, 128) as expected by tokenizer
    tokens = piano_roll_sequence_to_event_tokens(seg_transposed, max_seq_len=256)
    non_pad = tokens[tokens != 0]
    print(f"Token sequence length : {len(tokens)} (max_seq_len=256)")
    print(f"Non-padding tokens    : {len(non_pad)}")
    print(f"First 20 token IDs    : {tokens[:20].tolist()}")

## Step 5 — Run Full Dataset Processing

The function below processes all MAESTRO splits and writes `.npy` files to `data/processed/`.  
Skip this cell if the `.npy` files already exist.

In [ ]:
train_npy = PROCESSED_DATA_DIR / "maestro_train.npy"
if train_npy.exists():
    print("Processed data already found — loading to verify shapes.")
    for split in ("train", "validation", "test"):
        path = PROCESSED_DATA_DIR / f"maestro_{split}.npy"
        if path.exists():
            arr = np.load(path)
            print(f"  {path.name}: {arr.shape}  dtype={arr.dtype}")
else:
    print("Running full preprocessing (this may take several minutes)...")
    from src.preprocessing.midi_parser import process_maestro_dataset
    process_maestro_dataset()